# Test inference — eta product transformer

Loads a checkpoint and dataset, runs greedy + beam search (with brute-force coefficient recovery) on a dataset example, with an option to provide custom coefficients.

## 1. Imports & config

In [21]:
import pickle
import numpy as np
import torch
from fractions import Fraction

from transformer_eta_product import (
    EtaProductTransformer, EtaProductVocabulary, ModelConfig, SyntaxMasker,
)
from dataset_generator import build_phi_cache
from analyze_prediction import beam_search, bruteforce_coefficients, compare_formulas

# ---- paths ----
CKPT_PATH = './checkpoints_T_2/checkpoint_last.pt'
DATA_PATH = './eta_product_dataset_T_2.pkl'

# ---- inference config ----
SPLIT       = 'test'   # 'train' | 'val' | 'test'
EXAMPLE_IDX = 1
BEAM_WIDTH  = 20
MAX_LEN     = 80
USE_MASK    = True     # syntax masking during beam search

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cpu


## 2. Load dataset

In [24]:
with open(DATA_PATH, 'rb') as f:
    data = pickle.load(f)

dataset_cfg = data.get('config', {})

split = data[SPLIT]
n_coeffs = dataset_cfg.get('n_coeffs', 50)

## 3. Load checkpoint & rebuild vocab

The checkpoint stores `model_state_dict` + `config` + vocab metadata (`vocab_denominators`, `vocab_max_exp`, `vocab_max_num_coeff`, `vocab_max_shift`). We rebuild the `EtaProductVocabulary` from those — same logic as `analyze_prediction.py`, including the auto-fix loop in case some metadata is missing.

In [25]:
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
config = ckpt['config']
ckpt_args = ckpt.get('args', {})

denominators = ckpt.get('vocab_denominators',
               ckpt_args.get('denominators',
               dataset_cfg.get('denominators', [1])))
max_exp = ckpt.get('vocab_max_exp', None)
if max_exp is None:
    max_exp = max(abs(dataset_cfg.get('min_exp', -6)), abs(dataset_cfg.get('max_exp', 6)))
max_k = ckpt_args.get('max_k', dataset_cfg.get('max_k', 12))
max_num_coeff = ckpt.get('vocab_max_num_coeff',
                ckpt_args.get('max_num_coeff', dataset_cfg.get('max_num_coeff', 1)))
max_shift = ckpt.get('vocab_max_shift',
            ckpt_args.get('max_shift', dataset_cfg.get('max_shift', 0)))

vocab = EtaProductVocabulary(
    denominators=denominators, max_k=max_k, max_exp=max_exp,
    max_num_coeff=max_num_coeff, max_shift=max_shift,
)

# Auto-fix vocab size mismatch (same as analyze_prediction.py)
target_size = ckpt['model_state_dict']['token_embedding.weight'].shape[0]
if vocab.vocab_size != target_size:
    print(f'⚠ vocab mismatch: {vocab.vocab_size} vs {target_size}, searching...')
    found = False
    for try_mc in [1, 2, 3, 5, 10, max_num_coeff]:
        for try_ms in [0, 5, 10, max_shift]:
            for try_me in [max_exp, 6, 12, 24]:
                v = EtaProductVocabulary(
                    denominators=denominators, max_k=max_k, max_exp=try_me,
                    max_num_coeff=try_mc, max_shift=try_ms,
                )
                if v.vocab_size == target_size:
                    vocab = v
                    max_exp, max_num_coeff, max_shift = try_me, try_mc, try_ms
                    found = True
                    break
            if found: break
        if found: break
    if not found:
        raise RuntimeError(f'Cannot match vocab size {target_size}')

print(f'Vocab size: {vocab.vocab_size}')
print(f'  denominators={denominators}, max_k={max_k}, max_exp={max_exp}, '
      f'max_num_coeff={max_num_coeff}, max_shift={max_shift}')

Vocab size: 54
  denominators=[1], max_k=12, max_exp=6, max_num_coeff=20, max_shift=0


In [26]:
model = EtaProductTransformer(config, vocab)
model.load_state_dict(ckpt['model_state_dict'])
model.to(DEVICE).eval()

masker = SyntaxMasker(vocab) if USE_MASK else None

n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded — {n_params:,} parameters')
print(f'Epoch: {ckpt.get("epoch", "?")}')

Model loaded — 8,848,182 parameters
Epoch: 1600


## 4. Build phi cache (for brute-force coefficient recovery)

In [27]:
ds_denoms = dataset_cfg.get('denominators', denominators)
ds_max_k = dataset_cfg.get('max_k', max_k)
ds_max_num_coeff = dataset_cfg.get('max_num_coeff', max_num_coeff)

print(f'Building phi cache (denoms={ds_denoms}, max_k={ds_max_k}, max_order={n_coeffs + 20})...')
phi_cache = build_phi_cache(ds_denoms, max_k=ds_max_k, max_order=n_coeffs + 20)
print(f'✓ Cached {len(phi_cache["pos"])} phi functions')

Building phi cache (denoms=[1], max_k=12, max_order=120)...


Pre-computing phi: 100%|███████████████████████████████████████████████████████████████████████| 12/12 [00:00<00:00, 360.34it/s]

✓ Cached 12 phi functions


## 5. Inference helper

Runs greedy + beam, then brute-forces coefficients on every predicted skeleton.

In [47]:
def run_inference(coeffs_np, beam_width=BEAM_WIDTH, max_len=MAX_LEN,
                  use_mask=USE_MASK, verbose=True):
    """Run greedy + beam search + brute-force coefficient recovery on a q-series.
    
    Args:
        coeffs_np: 1D numpy array of integer Fourier coefficients (length n_coeffs).
    Returns:
        dict with 'greedy_formula', 'greedy_bf', 'beam' (list).
    """
    coeffs_np = np.asarray(coeffs_np, dtype=np.float64)
    coeffs_t = torch.tensor(coeffs_np, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    
    # ---- greedy ----
    greedy_idx = model.generate_greedy(coeffs_t, max_len=max_len)
    greedy_tokens = vocab.decode(greedy_idx)
    greedy_formula = vocab.tokens_to_formula(greedy_tokens)
    greedy_bf = bruteforce_coefficients(
        greedy_formula, coeffs_np, phi_cache,
        max_coeff=ds_max_num_coeff, n_coeffs=n_coeffs,
    )
    
    if verbose:
        print('--- GREEDY ---')
        if greedy_formula:
            print('skeleton:', vocab.formula_to_string(greedy_formula))
        else:
            print('INVALID SYNTAX:', greedy_tokens)
        if greedy_bf:
            print(f'after BF: {vocab.formula_to_string(greedy_bf["formula"])}')
            print(f'  coeffs found: {greedy_bf["coeffs_found"]}')
            print(f'  match: {greedy_bf["n_match"]}/{greedy_bf["n_total"]}'
                  f'  (all_match={greedy_bf["all_match"]})')
    
    # ---- beam ----
    beam_masker = SyntaxMasker(vocab) if use_mask else None
    beams = beam_search(model, coeffs_t, vocab, beam_width=beam_width,
                        max_len=max_len, masker=beam_masker)
    
    beam_results = []
    for rank, (b_idx, b_lp) in enumerate(beams):
        b_tokens = vocab.decode(b_idx)
        b_formula = vocab.tokens_to_formula(b_tokens)
        b_bf = bruteforce_coefficients(
            b_formula, coeffs_np, phi_cache,
            max_coeff=ds_max_num_coeff, n_coeffs=n_coeffs,
        )
        beam_results.append({
            'rank': rank, 'log_prob': b_lp,
            'formula': b_formula, 'bf': b_bf,
        })
    
    if verbose:
        print(f'\n--- BEAM (width={beam_width}, mask={use_mask}) ---')
        for br in beam_results:
            mark = '✓' if (br['bf'] and br['bf']['all_match']) else ' '
            if br['formula']:
                fs = vocab.formula_to_string(br['formula'])
                line = f'  {mark} [{br["rank"]}] lp={br["log_prob"]:.2f}  {fs}'
                if br['bf'] and br['bf']['all_match']:
                    line += f'  → coeffs={br["bf"]["coeffs_found"]}'
                print(line)
            else:
                print(f'    [{br["rank"]}] INVALID')
    
    return {
        'greedy_formula': greedy_formula,
        'greedy_bf': greedy_bf,
        'beam': beam_results,
    }

## 6. Run on a dataset example

In [51]:
#EXAMPLE_IDX = 13 #13 provides an example where exact match only appear for beam_width = 50 
#BEAM_WIDTH  = 50

EXAMPLE_IDX = 1
BEAM_WIDTH  = 10

coeffs_np = split['coefficients'][EXAMPLE_IDX].astype(np.float64)
true_formula = split['formulas'][EXAMPLE_IDX]
# Normalize legacy (factors-only) format
if 'sum_terms' not in true_formula:
    true_formula = {'sum_terms': [{'coeff': 1, 'shift': 0, 'factors': true_formula['factors']}]}

print(f'=== example [{EXAMPLE_IDX}] from {SPLIT} ===')
print(f'TRUE: {vocab.formula_to_string(true_formula)}')
print(f'coeffs[0:10] = {coeffs_np[:10].astype(int).tolist()}')
print()

results = run_inference(coeffs_np, beam_width=BEAM_WIDTH)

# Exact match check
print('\n=== summary ===')
if results['greedy_bf'] and results['greedy_bf']['all_match']:
    cmp = compare_formulas(results['greedy_bf']['formula'], true_formula)
    print(f"greedy exact: {cmp['exact']}")
else:
    print('greedy exact: False (no full coeff match)')

beam_exact = False
for br in results['beam']:
    if br['bf'] and br['bf']['all_match']:
        cmp = compare_formulas(br['bf']['formula'], true_formula)
        if cmp['exact']:
            beam_exact = True
            print(f"beam exact at rank {br['rank']}")
            break
if not beam_exact:
    print('beam exact: False')

=== example [1] from test ===
TRUE: η(q^4)^3 · η(q^6)^3 · η(q^10)^2 · η(q^11)^2 + 6 · (η(q^6) · η(q^8) · η(q^9)^3 · η(q^11)^5)
coeffs[0:10] = [1, 6, 0, 0, -3, 0, -3, -6, 0, -6]

--- GREEDY ---
skeleton: ? · (η(q^4)^3 · η(q^6)^3 · η(q^10)^2 · η(q^11)^2) + ? · (η(q^6) · η(q^8) · η(q^9)^3 · η(q^11)^5)
after BF: η(q^4)^3 · η(q^6)^3 · η(q^10)^2 · η(q^11)^2 + 6 · (η(q^6) · η(q^8) · η(q^9)^3 · η(q^11)^5)
  coeffs found: [1, 6]
  match: 100/100  (all_match=True)

--- BEAM (width=10, mask=True) ---
  ✓ [0] lp=-0.00  ? · (η(q^4)^3 · η(q^6)^3 · η(q^10)^2 · η(q^11)^2) + ? · (η(q^6) · η(q^8) · η(q^9)^3 · η(q^11)^5)  → coeffs=[1, 6]
    [1] lp=-7.95  ? · (η(q^4)^3 · η(q^6)^3 · η(q^10)^3 · η(q^12)) + ? · (η(q^6) · η(q^8) · η(q^9)^3 · η(q^11)^5)
    [2] lp=-11.77  ? · (η(q^4)^3 · η(q^6)^3 · η(q^10)^2 · η(q^11)^2) + ? · (η(q^6)^2 · η(q^9)^2 · η(q^11)^6)
    [3] lp=-12.03  ? · (η(q^4)^3 · η(q^6)^2 · η(q^9)^2 · η(q^10)^3) + ? · (η(q^6) · η(q^8) · η(q^9)^3 · η(q^11)^5)
    [4] lp=-12.05  ? · (η(q^4)^3 · η

## 7. User-provided coefficients

Edit the list below. Must be of length `n_coeffs` (or longer — only first `n_coeffs` are used). Integer coefficients of the q-series, starting from the lowest power (after shifting so the first nonzero coefficient is at index 0).

In [52]:
# example: paste your own q-series coefficients here.
# convenience: pad with zeros if shorter than n_coeffs.
# note however that the model has not been trained with padding so this is unlikely to work.
user_coeffs = [1.0, 0.0, 0.0, -2.0, -6.0, 4.0, 8.0, 12.0, 7.0, -16.0, 6.0, -38.0, -58.0, -12.0, -54.0, 96.0, 36.0, 136.0, 186.0, -96.0, 164.0, -196.0, 6.0, -212.0, -398.0, 48.0, -258.0, 386.0, -156.0, 128.0, 204.0, 300.0, 249.0, -272.0, 48.0, -72.0, 708.0, 120.0, -22.0, -1376.0, 60.0, -264.0, -864.0, -432.0, -854.0, 1924.0, -216.0, 2092.0, 1269.0, -864.0, 2054.0, 1254.0, 426.0, -2792.0, -3148.0, 84.0, -1935.0, -256.0, 630.0, -2176.0, -266.0, 1116.0, 454.0, -4064.0, 432.0, 4368.0, 3698.0, 912.0, 1178.0, 396.0, -1800.0, 4616.0, 396.0, 624.0, -2406.0, 4216.0, -2076.0, -5612.0, 4712.0, -1632.0, 1369.0, -6512.0, -252.0, -8674.0, -6856.0, -1212.0, -1160.0, 2752.0, 1116.0, 9304.0, -1332.0, 528.0, 4798.0, 9016.0, 4842.0, 1236.0, -5397.0, -144.0,-3746.0,5666.0]

user_coeffs = np.array(user_coeffs, dtype=np.float64)
if len(user_coeffs) < n_coeffs:
    user_coeffs = np.pad(user_coeffs, (0, n_coeffs - len(user_coeffs)))
    print(f'⚠ Padded with zeros to length {n_coeffs}. '
          f'For best results, provide the full {n_coeffs}-coefficient q-series.')
elif len(user_coeffs) > n_coeffs:
    user_coeffs = user_coeffs[:n_coeffs]

user_results = run_inference(user_coeffs, beam_width=BEAM_WIDTH)

--- GREEDY ---
skeleton: ? · (η(q^2)^2 · η(q^3)^2 · η(q^4)^5 · η(q^6)^3) + ? · (η(q^6)^6 · η(q^10)^6)
after BF: η(q^2)^2 · η(q^3)^2 · η(q^4)^5 · η(q^6)^3 + 2 · (η(q^6)^6 · η(q^10)^6)
  coeffs found: [1, 2]
  match: 100/100  (all_match=True)

--- BEAM (width=10, mask=True) ---
  ✓ [0] lp=-0.00  ? · (η(q^2)^2 · η(q^3)^2 · η(q^4)^5 · η(q^6)^3) + ? · (η(q^6)^6 · η(q^10)^6)  → coeffs=[1, 2]
    [1] lp=-6.35  ? · (η(q^2)^2 · η(q^3)^2 · η(q^4)^5 · η(q^6)^3) + ? · (η(q^6)^6 · η(q^9) · η(q^10)^4 · η(q^11))
    [2] lp=-8.14  ? · (η(q^2)^2 · η(q^3)^2 · η(q^4)^5 · η(q^6)^3) + ? · (η(q^6)^6 · η(q^9)^2 · η(q^10)^3 · η(q^12))
    [3] lp=-8.61  ? · (η(q^2)^2 · η(q^3)^2 · η(q^4)^5 · η(q^6)^3) + ? · (η(q^6)^5 · η(q^9)^4 · η(q^10)^3)
    [4] lp=-10.33  ? · (η(q^2)^2 · η(q^3)^2 · η(q^4)^5 · η(q^6)^3) + ? · (η(q^6)^6 · η(q^9)^3 · η(q^11)^3)
    [5] lp=-11.34  ? · (η(q^2)^2 · η(q^3)^2 · η(q^4)^5 · η(q^6)^3) + ? · (η(q^6)^6 · η(q^7) · η(q^10)^2 · η(q^11)^3)
    [6] lp=-11.54  ? · (η(q^2)^2 · η(q^3)^2 · η(q^4